# 🍦 Ice Cream Sales Predictor, version 2

**Homework 3:** add at least one more predictor, retrain the model, save it, and use it in the app.

Version 1 (class example) predicted cones sold from **temperature only**. Version 2 adds two things that also drive sales:

- **Weekend:** people have free time, so more cones sell on Saturday and Sunday.
- **Chance of rain:** rainy days keep people indoors, so fewer cones sell.

Both new predictors are things the app can know automatically: the weekend comes from the date, and the chance of rain comes from the same free weather forecast as the temperature.

Like the class example, the data is **made up** to show the steps.

## 1. Make the data
One year of days. Temperature between 50 and 100 °F, a weekend flag (1 = Saturday or Sunday), and the chance of rain (0 to 100%).

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 365

df = pd.DataFrame({
    "temperature": rng.uniform(50, 100, n).round(1),
    "is_weekend": (np.arange(n) % 7 >= 5).astype(int),
    "rain_chance": rng.integers(0, 101, n),
})

# Made-up rule for how many cones sell, plus random noise
df["sales"] = (5 * df["temperature"] - 100
               + 60 * df["is_weekend"]
               - 1.2 * df["rain_chance"]
               + rng.normal(0, 20, n)).round().clip(lower=0).astype(int)

df.head()

   temperature  is_weekend  rain_chance  sales
0         88.7           0           73    259
1         71.9           0           42    214
2         92.9           0           93    284
3         84.9           0           78    224
4         54.7           0           21    129

In [2]:
df.describe().round(1)

       temperature  is_weekend  rain_chance  sales
count        365.0       365.0        365.0  365.0
mean          74.6         0.3         50.3  228.0
std           14.3         0.5         29.4   87.9
min           50.4         0.0          0.0    8.0
25%           61.6         0.0         24.0  166.0
50%           74.6         0.0         51.0  226.0
75%           87.4         1.0         75.0  294.0
max           99.6         1.0        100.0  468.0

## 2. Look at the data
Does each predictor seem to move sales?

In [3]:
import plotly.express as px

px.scatter(df, x="temperature", y="sales", color=df["is_weekend"].map({0: "weekday", 1: "weekend"}),
           title="Hotter days and weekends sell more")

In [4]:
px.scatter(df, x="rain_chance", y="sales", title="Rainier days sell fewer cones")

## 3. Version 1: temperature only (the class model)
We retrain it here so we can compare.

In [5]:
from sklearn.linear_model import LinearRegression

model_v1 = LinearRegression()
model_v1.fit(df[["temperature"]], df["sales"])
print("R2, temperature only:", round(model_v1.score(df[["temperature"]], df["sales"]), 3))

R2, temperature only: 0.701


## 4. Version 2: temperature + weekend + chance of rain

In [6]:
FEATURES = ["temperature", "is_weekend", "rain_chance"]

X = df[FEATURES]
y = df["sales"]

model = LinearRegression()
model.fit(X, y)

print("R2, three predictors:", round(model.score(X, y), 3))
for name, coef in zip(FEATURES, model.coef_):
    print(f"  {name}: {coef:.2f}")
print(f"  intercept: {model.intercept_:.2f}")

R2, three predictors: 0.947
  temperature: 5.00
  is_weekend: 59.29
  rain_chance: -1.20
  intercept: -101.76


**Reading the result:** each extra degree adds about 5 cones, a weekend adds about 60, and each extra 10% chance of rain removes about 12. Adding the two predictors raises R² (more of the day-to-day ups and downs are explained).

## 5. Test a prediction
The new row must use the **same column names in the same order** as training.

In [7]:
tomorrow = pd.DataFrame([[85, 1, 20]], columns=FEATURES)   # 85 °F, Saturday, 20% chance of rain
model.predict(tomorrow)[0]

np.float64(358.7881615204267)

## 6. Save the model
This writes `model.pkl`. In Colab, the second line downloads it to your computer so you can upload it to GitHub.

In [8]:
import pickle

with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# In Colab only:
# from google.colab import files
# files.download("model.pkl")

In [9]:
# Check the saved file works
with open("model.pkl", "rb") as f:
    loaded = pickle.load(f)
loaded.predict(tomorrow)[0]

np.float64(358.7881615204267)

## 7. Write the app files
Copy these into GitHub **without** the `%%writefile` line.

`requirements.txt` pins scikit-learn to the version that saved the model, so Streamlit Cloud loads it with the same version.

In [10]:
import sklearn
print(sklearn.__version__)

1.9.1


In [11]:
%%writefile app.py
import pickle
from datetime import date, timedelta

import pandas as pd
import requests
import streamlit as st

# ---- Settings ----------------------------------------------------------
CITIES = {
    "Pittsburgh": (40.44, -79.99),
    "New York": (40.71, -74.01),
    "Chicago": (41.88, -87.63),
    "Miami": (25.76, -80.19),
    "Los Angeles": (34.05, -118.24),
}
FEATURES = ["temperature", "is_weekend", "rain_chance"]   # same names and order as in training
LOW_TEMP, HIGH_TEMP = 50, 100        # the temperatures (F) the model learned from
LOW_SALES, HIGH_SALES = 200, 300     # below LOW_SALES is a slow day, above HIGH_SALES is a busy one
URL = "https://api.open-meteo.com/v1/forecast"
# ------------------------------------------------------------------------

# Load the trained model (we trained it once, in Colab, and saved it to a file)
with open("model.pkl", "rb") as f:
    model = pickle.load(f)

st.set_page_config(page_title="Ice Cream Sales Predictor", page_icon="🍦")
st.title("🍦 Ice Cream Sales Predictor")
st.caption("Pick a city and a date. The app gets the forecast (high temperature and chance of rain), "
           "checks if it is a weekend, and predicts how many cones you will sell.")

city = st.selectbox("City", list(CITIES))
selected_date = st.date_input(
    "Date",
    value=date.today() + timedelta(days=1),
    min_value=date.today(),
    max_value=date.today() + timedelta(days=15),    # the forecast reaches about 16 days ahead
)

if st.button("Predict sales", type="primary"):
    # 1. Ask the weather API for the forecast high and the chance of rain
    latitude, longitude = CITIES[city]
    response = requests.get(
        URL,
        params={
            "latitude": latitude,
            "longitude": longitude,
            "daily": "temperature_2m_max,precipitation_probability_max",
            "temperature_unit": "fahrenheit",
            "timezone": "auto",
            "start_date": str(selected_date),
            "end_date": str(selected_date),
        },
        timeout=30,
    )
    response.raise_for_status()
    daily = response.json()["daily"]
    temperature = daily["temperature_2m_max"][0]
    rain_chance = daily["precipitation_probability_max"][0]
    if rain_chance is None:              # far-off days sometimes have no rain forecast yet
        rain_chance = 0
        st.caption("No rain forecast for this day yet, so the app assumes 0% chance of rain.")

    # 2. Weekend or weekday? Saturday is 5 and Sunday is 6
    is_weekend = 1 if selected_date.weekday() >= 5 else 0

    # 3. Give the model the same inputs it was trained on, in the same order, then predict
    features = pd.DataFrame([[temperature, is_weekend, rain_chance]], columns=FEATURES)
    prediction = max(0, model.predict(features)[0])

    # 4. Show the result
    col1, col2, col3 = st.columns(3)
    col1.metric("Forecast high", f"{temperature:.0f} °F")
    col2.metric("Chance of rain", f"{rain_chance:.0f}%")
    col3.metric("Day", "Weekend" if is_weekend else "Weekday")
    st.metric("Predicted sales", f"{prediction:.0f} cones")

    if temperature < LOW_TEMP or temperature > HIGH_TEMP:
        st.warning(f"This temperature is outside the range the model learned from ({LOW_TEMP} to {HIGH_TEMP} °F). "
                   "Treat the number with caution.")

    if prediction > HIGH_SALES:
        st.info("Busy day: make sure you are fully stocked.")
    elif prediction > LOW_SALES:
        st.info("A normal day is expected.")
    else:
        st.info("A slow day: consider a promotion, or order less.")

    with st.expander("How the model decides"):
        st.write(
            f"Each extra degree adds about **{model.coef_[0]:.1f} cones**. "
            f"A weekend adds about **{model.coef_[1]:.0f} cones**. "
            f"Each extra 10% chance of rain changes sales by about **{model.coef_[2] * 10:.0f} cones**."
        )


Writing app.py


In [12]:
%%writefile requirements.txt
streamlit
pandas
requests
scikit-learn==1.9.1


Writing requirements.txt


## 8. Deploy
1. Create a public GitHub repo with `app.py`, `model.pkl`, `requirements.txt` at the top level (branch `main`).
2. On share.streamlit.io, create an app from the repo, main file `app.py`.
3. Open the link, pick a city and a date, and click **Predict sales**.

No API key is needed: Open-Meteo is free.